# ForestGuard

## 05 — Modelos de clasificación

Entrenamiento y comparación de modelos de clasificación con el dataset procesado.

> **Nota:** Este notebook corresponde a una etapa preliminar de ForestGuard.
>
> La variable objetivo utilizada representa el nivel de afectación de incendios ya registrados, por lo que no debe interpretarse como una predicción de la probabilidad de que ocurra un incendio.
>
> El modelo principal de riesgo se desarrolla posteriormente utilizando una variable objetivo binaria `incendio` (`0/1`).

## 1. Carga del dataset procesado

In [ ]:
from pathlib import Path
from google.colab import files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay
)

ROOT = Path("/content")
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

archivo_processed = PROCESSED / "incendios_forestales_2022_2025_limpio.csv"

if not archivo_processed.exists():
    print("Sube el archivo generado por el módulo 02:")
    print("- incendios_forestales_2022_2025_limpio.csv")

    subidos = files.upload()

    for nombre, contenido in subidos.items():
        with open(PROCESSED / nombre, "wb") as f:
            f.write(contenido)

if not archivo_processed.exists():
    raise FileNotFoundError(
        "No se encontró el dataset procesado. "
        "Ejecuta el módulo 02 o sube el CSV generado."
    )

df = pd.read_csv(archivo_processed)
print("Dataset cargado:", df.shape)

## 2. Revisión de la variable objetivo

Antes de entrenar los modelos se revisa cuántos registros existen en cada nivel.

Los registros sin `nivelinf` se excluyen únicamente de esta tarea de clasificación porque no poseen una etiqueta conocida.

In [ ]:
objetivo = "nivelinf"

display(
    df[objetivo]
    .value_counts(dropna=False)
    .to_frame("Cantidad")
)

datos_modelo = df.dropna(subset=[objetivo]).copy()

print(
    "Registros disponibles para clasificación:",
    datos_modelo.shape[0]
)

## 3. Selección de variables

Se incluyen variables temporales, geográficas y de impacto disponibles.

`hadecobert` se excluye del conjunto de entrada para reducir el riesgo de fuga de información.

In [ ]:
variables_numericas = [
    "año",
    "mes",
    "dia",
    "latitud",
    "longitud",
    "personas_afectadas",
    "familias_afectadas",
    "animales_afectados",
    "animales_muertos",
]

variables_categoricas = [
    "provincia",
    "canton",
    "region",
]

variables_numericas = [
    c for c in variables_numericas
    if c in datos_modelo.columns
]

variables_categoricas = [
    c for c in variables_categoricas
    if c in datos_modelo.columns
]

variables = variables_numericas + variables_categoricas

X = datos_modelo[variables]
y = datos_modelo[objetivo].astype(str)

print("Variables utilizadas:", variables)

## 4. División de entrenamiento y prueba

Se utiliza estratificación cuando todas las clases tienen suficientes observaciones. Esto ayuda a mantener una distribución parecida de clases entre entrenamiento y prueba.

In [ ]:
conteo_clases = y.value_counts()

usar_stratify = (
    len(conteo_clases) > 1
    and conteo_clases.min() >= 2
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=10,
    stratify=y if usar_stratify else None
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Estratificación utilizada:", usar_stratify)

## 5. Preprocesamiento

In [ ]:
preprocesamiento_numerico = Pipeline([
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalado", StandardScaler())
])

preprocesamiento_categorico = Pipeline([
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocesador = ColumnTransformer([
    (
        "numericas",
        preprocesamiento_numerico,
        variables_numericas
    ),
    (
        "categoricas",
        preprocesamiento_categorico,
        variables_categoricas
    )
])

## 6. Modelos implementados

Se comparan dos algoritmos:

1. **Regresión Logística:** modelo lineal de clasificación.
2. **Random Forest Classifier:** modelo basado en múltiples árboles de decisión.

También se utiliza `DummyClassifier` como baseline para conocer el desempeño de una estrategia muy básica basada en la clase más frecuente.

In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=10
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=10,
        n_jobs=-1
    ),
    "Baseline (clase mayoritaria)": DummyClassifier(
        strategy="most_frequent"
    )
}

## 7. Entrenamiento y comparación de métricas

Debido al posible desbalance de clases, además de accuracy se utilizan métricas macro y balanced accuracy.

In [ ]:
resultados = []
modelos_entrenados = {}
predicciones_modelos = {}

for nombre, modelo in modelos.items():

    pipeline = Pipeline([
        ("preprocesamiento", preprocesador),
        ("modelo", modelo)
    ])

    pipeline.fit(X_train, y_train)

    predicciones = pipeline.predict(X_test)

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(
            y_test,
            predicciones
        ),
        "Precision macro": precision_score(
            y_test,
            predicciones,
            average="macro",
            zero_division=0
        ),
        "Recall macro": recall_score(
            y_test,
            predicciones,
            average="macro",
            zero_division=0
        ),
        "F1 macro": f1_score(
            y_test,
            predicciones,
            average="macro",
            zero_division=0
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_test,
            predicciones
        )
    })

    modelos_entrenados[nombre] = pipeline
    predicciones_modelos[nombre] = predicciones

resultados_clasificacion = pd.DataFrame(resultados)
resultados_clasificacion = (
    resultados_clasificacion
    .sort_values("F1 macro", ascending=False)
)

display(resultados_clasificacion.round(4))

## 8. Comparación gráfica

In [ ]:
grafico = (
    resultados_clasificacion
    .set_index("Modelo")[
        ["Accuracy", "F1 macro", "Balanced accuracy"]
    ]
)

grafico.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Comparación de modelos de clasificación")
plt.xlabel("Modelo")
plt.ylabel("Puntuación")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 9. Selección preliminar del mejor modelo

Se utiliza F1 macro como criterio principal porque da el mismo peso a cada clase y resulta más informativo cuando existe desbalance.

In [ ]:
modelos_reales = resultados_clasificacion[
    ~resultados_clasificacion["Modelo"].str.contains("Baseline")
].copy()

mejor_modelo_clasificacion = (
    modelos_reales
    .sort_values("F1 macro", ascending=False)
    .iloc[0]
)

print("Mejor modelo de clasificación:")
print(mejor_modelo_clasificacion)

## 10. Reporte y matriz de confusión del mejor modelo

In [ ]:
nombre_mejor = mejor_modelo_clasificacion["Modelo"]
pred_mejor = predicciones_modelos[nombre_mejor]

print(
    classification_report(
        y_test,
        pred_mejor,
        zero_division=0
    )
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_mejor,
    xticks_rotation=45
)

plt.title(
    f"Matriz de confusión — {nombre_mejor}"
)

plt.tight_layout()
plt.show()

## 11. Conclusión

La ampliación 2022–2025 permite entrenar los modelos con una mayor cantidad de información y comparar dos algoritmos de clasificación.

El baseline permite comprobar si los modelos reales superan una estrategia básica. Debido al posible desbalance de `nivelinf`, las métricas macro y balanced accuracy son especialmente importantes para interpretar los resultados.

# Resultado final

Al finalizar la ejecución completa, el flujo habrá:

- obtenido datos de incendios forestales de 2022–2025;
- integrado y limpiado las fuentes;
- generado el dataset procesado;
- realizado análisis exploratorio;
- comparado modelos de regresión;
- comparado modelos de clasificación.

Los resultados y métricas deben interpretarse con base en la ejecución real del notebook.